# 03. 한국어 어휘 확장 + 추가학습 + 평가 (A100, 시간은 config의 학습량에 따라)
`ko-k32`(전문가 32개)를 한국어로 개조합니다.
1. **어휘 확장 (CPU, 수 분)**: 한국어 BPE를 학습해 한국어 토큰 `vocab.n_new`개를 덧붙입니다. 같은 한국어를 약 40% 적은 토큰으로 처리합니다.
2. **1단계 학습**: 모델은 고정하고 새 토큰 임베딩만 한국어 위키로 학습합니다.
3. **2단계 학습**: 한국어 지시문(KULLM-v2 6천 개)으로 LoRA (어텐션 + 공유 전문가 + 남은 전문가, 라우터 고정) + 새 토큰 임베딩.
4. 합쳐서 GGUF(Q6_K, Q4_K_M, Q3_K_M)로 만들고 바로 평가합니다. Q4_K_M·Q3_K_M은 드라이브에 저장해 04 노트북(T4·L4)과 내 PC에서 잽니다.

학습 중간 저장은 드라이브(`cfinetune/ckpt/`)에 하므로, 세션이 끊기면 **처음부터 다시 실행**하면 이어서 학습합니다.
드라이브에 약 15GB 여유가 필요합니다 (학습 중간 저장 + 최종 모델 Q4_K_M·Q3_K_M).

In [ ]:
# 저장소 받기 + 패키지 설치
BRANCH = "ccr-a463bea6-bpbyrh"   # main에 합친 뒤에는 "main"
REPO = "github.com/b1bibig/Cfinetune"
import os
try:  # 저장소가 비공개면 코랩 왼쪽 열쇠(보안 비밀)에 GH_TOKEN 을 넣어 두세요
    from google.colab import userdata
    _tok = userdata.get("GH_TOKEN")
    REPO_URL = f"https://{_tok}@{REPO}"
except Exception:
    REPO_URL = f"https://{REPO}"
if not os.path.exists("/content/Cfinetune"):
    !git clone -q -b {BRANCH} {REPO_URL} /content/Cfinetune
%cd /content/Cfinetune
!git pull -q
%pip install -q -r requirements-colab.txt

In [ ]:
# 구글 드라이브: 결과(results/)를 세션이 끊겨도 남도록 저장
from google.colab import drive
drive.mount("/content/drive")
import os, shutil, sys
DRIVE = "/content/drive/MyDrive/cfinetune"
os.makedirs(DRIVE, exist_ok=True)

def pull_results():
    if os.path.exists(f"{DRIVE}/results"):
        shutil.copytree(f"{DRIVE}/results", "results", dirs_exist_ok=True)

def push_results():
    shutil.copytree("results", f"{DRIVE}/results", dirs_exist_ok=True)
    print("드라이브에 결과 저장함")

pull_results()
sys.path.insert(0, "scripts"); sys.path.insert(0, "src")

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU 메모리 {mem:.0f}GB")
assert mem >= 38, "런타임 > 런타임 유형 변경에서 A100 를 고르세요"

In [ ]:
!bash scripts/build_llama_cpp.sh /content/llama.cpp {DRIVE}/cache

In [ ]:
from _common import resolve_model_dir, load_config
cfg = load_config()
base, VX = cfg["lora"]["base_variant"], cfg["vocab"]["enabled"]
NAME = f"{base}{'-vx' if VX else ''}-lora"
TRAIN_IN = f"/content/models/pruned/{base}{'-vx' if VX else ''}"
OUT, CKPT = f"/content/lora/{NAME}", f"{DRIVE}/ckpt/{NAME}"
QUANTS, DEPLOY = cfg["lora"]["quant_after"], cfg["deploy_quant"]

def done():
    return all(os.path.exists(f"results/eval/{NAME}-{q}.json") for q in QUANTS)

def merged():
    return os.path.exists(f"{OUT}/merged/config.json")

print(NAME, "- 평가까지 끝남" if done() else "")

## 1. 칼질 + 어휘 확장 (CPU)

In [ ]:
if not done() and not merged() and not os.path.exists(f"{TRAIN_IN}/config.json"):
    SRC = resolve_model_dir(cfg["model"]["id"])
    !python scripts/02_prune.py --variant {base} --src {SRC} --out-dir /content/models/pruned
    shutil.rmtree(SRC)   # 디스크 확보 (원본은 더 이상 필요 없음)
    if VX:
        !python scripts/02b_extend_vocab.py --src /content/models/pruned/{base} --dst {TRAIN_IN}
        shutil.rmtree(f"/content/models/pruned/{base}")
    push_results()

## (선택) 시범 학습: 속도와 메모리 재기 (약 10분)
각 단계를 5스텝만 돌려 `train_samples_per_second`와 최대 VRAM을 봅니다 (1단계는 x1024 하면 초당 토큰).
이 속도로 `configs/experiment.yaml`의 `vocab.stage1.n_tokens`를 정하세요.

In [ ]:
!nohup nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits -lms 1000 > /content/mem.log 2>&1 &
!python scripts/04_train_lora.py --model {TRAIN_IN} --out /content/pilot --ckpt-dir /content/pilot/ckpt --stage1-tokens 300000 --stage1-max-steps 5 --max-steps 5 --no-merge
!pkill nvidia-smi; echo "최대 VRAM(MiB):"; sort -n /content/mem.log | tail -1
shutil.rmtree("/content/pilot", ignore_errors=True)

## 2. 학습 (1단계 + 2단계, 끊기면 이 셀까지 다시 실행)

In [ ]:
if not done() and not merged():
    !python scripts/04_train_lora.py --model {TRAIN_IN} --out {OUT} --ckpt-dir {CKPT}
    if merged():
        shutil.copytree(f"{OUT}/adapter", f"{DRIVE}/lora/{NAME}/adapter", dirs_exist_ok=True)
        shutil.rmtree(f"{CKPT}/stage2", ignore_errors=True)   # 드라이브 공간 확보 (1단계 결과는 남김)
    push_results()

## 3. GGUF 변환 + 평가

In [ ]:
todo = [q for q in QUANTS if not os.path.exists(f"results/eval/{NAME}-{q}.json")]
if todo:
    QS = " ".join(todo)
    !python scripts/03_make_gguf.py --hf-dir {OUT}/merged --name {NAME} --out-dir /content/gguf --quant {QS}
    for q in todo:
        g = f"/content/gguf/{NAME}-{q}.gguf"
        !python scripts/05_eval.py --gguf {g} --name {NAME}-{q} --tokenizer {OUT}/merged
        push_results()
os.makedirs(f"{DRIVE}/gguf", exist_ok=True)
for q in cfg["lora"]["to_drive"]:   # 04 노트북(작은 GPU)과 내 PC 측정용
    g = f"/content/gguf/{NAME}-{q}.gguf"
    if os.path.exists(g) and not os.path.exists(f"{DRIVE}/gguf/{NAME}-{q}.gguf"):
        shutil.copy(g, f"{DRIVE}/gguf/")
        print("드라이브에 저장:", f"{DRIVE}/gguf/{NAME}-{q}.gguf")

끝나면 **런타임 → 런타임 연결 해제 및 삭제**를 눌러 유닛이 계속 나가지 않게 하세요.